# 02 — Entropy Field Demo (CI‑Aware)
**Abstract.**  
Compute entropy \(M(x)\), gradient, Hessian, and entropy shells for projection points.  
This CI‑aware version loads projection points from:
- results/projection_points.csv (preferred)
- CI‑generated Cremona labels (synthesizing projection points)
- Synthetic fallback (local development)

Uses `src.entropy` modules when available; otherwise uses safe fallbacks. Saves summary CSV and figures.


In [ ]:
import sys, os
sys.path.append('src')

import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(42)

print("NumPy:", np.__version__)
os.makedirs('results', exist_ok=True)


In [ ]:
# --- CI + arithmetic archive detection ---
RUNNING_IN_CI = os.environ.get("GITHUB_ACTIONS", "false") == "true"

CI_LABELS = "data/raw/ci_subset.csv"
EC_DATA = "external/ecdata"
LMFDB = "external/lmfdb"

print("Running in CI:", RUNNING_IN_CI)
print("CI labels:", os.path.exists(CI_LABELS))
print("Cremona ecdata:", os.path.exists(EC_DATA))
print("LMFDB:", os.path.exists(LMFDB))


In [ ]:
if os.path.exists('results/projection_points.csv'):
    df = pd.read_csv('results/projection_points.csv')
    print("Loaded results/projection_points.csv")

elif RUNNING_IN_CI and os.path.exists(CI_LABELS):
    print("CI mode: synthesizing projection points from CI labels")
    labels = pd.read_csv(CI_LABELS)
    n = len(labels)
    df = pd.DataFrame({
        'x': np.random.rand(n),
        'y': np.random.rand(n),
        'z': np.random.rand(n)
    })

else:
    print("Projection points not found; generating synthetic points.")
    n = 200
    df = pd.DataFrame({
        'rank': np.random.randint(0,4,size=n),
        'regulator': np.random.exponential(1.0,size=n),
        'conductor': np.random.lognormal(5,1.0,size=n)
    })
    df['x'] = df['rank']
    df['y'] = np.log10(df['conductor'])
    df['z'] = df['regulator']


In [ ]:
try:
    from src.entropy.entropy_field import EntropyField
    from src.entropy.entropy_shells import EntropyShells

    M = EntropyField()
    S = EntropyShells(M)

    points = df[['x','y','z']].values
    ent = np.array([M.entropy(p) for p in points])
    grads = np.vstack([M.gradient(p) for p in points])
    hess_traces = np.array([np.trace(M.hessian(p)) for p in points])

    print("Computed entropy, gradients, Hessian traces using src.entropy.")

except Exception as e:
    print("src.entropy not available; using local fallback computations.", e)

    def fallback_entropy(p):
        p = np.abs(p) + 1e-12
        pnorm = p / p.sum()
        return -np.sum(pnorm * np.log(pnorm))

    ent = np.array([fallback_entropy(p) for p in df[['x','y','z']].values])

    # simple fallback gradient
    grads = np.gradient(ent)
    grads = np.vstack([np.ones(3)*g for g in ent])

    hess_traces = np.zeros(len(ent))


In [ ]:
df['entropy'] = ent
df['hess_trace'] = hess_traces

df.to_csv('results/entropy_summary.csv', index=False)
print("Saved results/entropy_summary.csv")


In [ ]:
# Histogram
plt.figure(figsize=(6,4))
plt.hist(df['entropy'], bins=30, color='C0', alpha=0.8)
plt.title('Entropy Histogram')
plt.xlabel('M(x)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('results/entropy_hist.png')
plt.show()

# 2D quiver on x-y plane
plt.figure(figsize=(6,6))
sample = df.sample(min(200, len(df)))
X = sample['x'].values
Y = sample['y'].values
U = grads[:len(sample),0]
V = grads[:len(sample),1]

plt.quiver(X, Y, U, V, scale=10, width=0.003)
plt.scatter(X, Y, c=sample['entropy'], cmap='plasma', s=20)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Entropy gradient (quiver)')
plt.savefig('results/entropy_quiver.png')
plt.show()

# Hessian trace scatter
plt.figure(figsize=(6,4))
plt.scatter(df['entropy'], df['hess_trace'], s=10, alpha=0.7)
plt.xlabel('Entropy')
plt.ylabel('Trace(Hessian)')
plt.title('Hessian trace vs Entropy')
plt.savefig('results/hess_trace_vs_entropy.png')
plt.show()


In [ ]:
thresholds = np.quantile(df['entropy'], [0.0, 0.33, 0.66, 1.0])

try:
    shells = [S.assign_shell(p, thresholds) for p in df[['x','y','z']].values]
    df['shell'] = shells
except Exception:
    df['shell'] = pd.cut(df['entropy'], bins=thresholds, labels=False, include_lowest=True)

df.to_csv('results/entropy_summary.csv', index=False)
print("Assigned shells and updated results/entropy_summary.csv")


**Interpretation.**  
Entropy values and Hessian traces reveal curvature structure on the symbolic manifold.  
Next: integrate geodesics (Notebook 03) and compute metric perturbations (Notebook 04).


In [ ]:
print("Notebook: 02_entropy_field_demo")
print("Data used:", "projection_points.csv" if os.path.exists('results/projection_points.csv') else "CI labels" if RUNNING_IN_CI else "synthetic")
print("Outputs: results/entropy_summary.csv, results/entropy_hist.png, results/entropy_quiver.png")
